# Practice Day 10

##### Capstone — full pipeline across all datasets
##### End-to-end: clean → join → aggregate → window → export

##### Q1 Build a complete Titanic analysis pipeline using method chaining: load → join class_name → clean nulls → add computed columns (fare_per_person = Fare/(SibSp+Parch+1), age_group bins) → group by ClassName + Sex → survival rate + avg fare. All in one chained expression.
##### Q2 From salaries.csv, build a year-over-year salary report: clean nulls → group by Year + JobTitle → avg pay → unstack Year into columns → add a YoY growth % column for each title. Export to Excel with sheet name "YoY Report".
##### Q3 Using the demographics datasets (both combined), join income_group.xlsx and produce a summary pivot: rows = Income Group, columns = [avg Birth rate, avg Internet users, country count]. Apply memory optimization (convert Income Group to category dtype).
##### Q4 From restaurant.csv, write a fully method-chained pipeline that: filters weekends → adds tip_pct → groups by day + smoker → aggregates total revenue and avg tip_pct → sorts by total revenue desc → saves to CSV.
##### Q5 Combine any 3 datasets of your choice into a mini data warehouse analysis. Define your own question, build the pipeline, and document what you found. This is your open-ended challenge.

In [1]:
import pandas as pd
import numpy as np
titanic = pd.read_csv(r"C:\Users\user\Desktop\Data_Science_Project\DataSample\titanic.csv") 
class_nm = pd.read_excel(r"C:\Users\user\Desktop\Data_Science_Project\DataSample\class_name.xlsx")
salary = pd.read_csv(r"C:\Users\user\Desktop\Data_Science_Project\DataSample\salaries.csv")
display(titanic.head())
display(class_nm.head())
display(salary.head())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,Pclass_number,ClassName,TempField
0,1,First class,0.972901
1,2,Second class,0.707523
2,3,Third class,0.381333


,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,1,NATHANIEL FORD,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,167411.18,0.00,400184.25,NaN,567595.43,567595.43,2011,NaN,San Francisco,NaN
1,2,GARY JIMENEZ,CAPTAIN III (POLICE DEPARTMENT),155966.02,245131.88,137811.38,NaN,538909.28,538909.28,2011,NaN,San Francisco,NaN
2,3,ALBERT PARDINI,CAPTAIN III (POLICE DEPARTMENT),212739.13,106088.18,16452.60,NaN,335279.91,335279.91,2011,NaN,San Francisco,NaN
3,4,CHRISTOPHER CHONG,WIRE ROPE CABLE MAINTENANCE MECHANIC,77916.00,56120.71,198306.90,NaN,332343.61,332343.61,2011,NaN,San Francisco,NaN
4,5,PATRICK GARDNER,"DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)",134401.60,9737.00,182234.59,NaN,326373.19,326373.19,2011,NaN,San Francisco,NaN


In [86]:
# Q1 Build a complete Titanic analysis pipeline using method chaining: 
# load → join class_name → 
# clean nulls → 
# add computed columns (fare_per_person = Fare/(SibSp+Parch+1), age_group bins) 
# → group by ClassName + Sex → survival rate + avg fare. 
# All in one chained expression.

In [87]:
df_q1 = pd.merge(titanic,class_nm, left_on = 'Pclass', right_on = 'Pclass_number').dropna()

In [88]:
df_q1['fare_per_person'] = df_q1['Fare']/(df_q1['SibSp']+df_q1['Parch']+1)

In [89]:
df_q1.groupby(['ClassName','Sex'])[['Survived','Fare']].mean()

Survived        Fare
ClassName    Sex                         
First class  female  0.959459  103.128209
             male    0.416667   75.957888
Second class female  0.888889   14.865744
             male    0.666667   23.812500
Third class  female  0.600000   13.360000
             male    0.400000    8.695000

In [90]:
result_q1 = (
    pd.merge(titanic, class_nm, left_on='Pclass', right_on='Pclass_number')
    .dropna(subset=['Age'])                          # clean nulls in Age only
    .assign(
        fare_per_person = lambda x: x['Fare'] / (x['SibSp'] + x['Parch'] + 1),
        age_group = lambda x: pd.cut(
            x['Age'],
            bins=[0, 12, 18, 35, 60, 100],
            labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
        )
    )
    .groupby(['ClassName', 'Sex'])[['Survived', 'Fare']]
    .mean()
)
result_q1

Survived        Fare
ClassName    Sex                         
First class  female  0.964706  107.946275
             male    0.396040   71.142781
Second class female  0.918919   21.951070
             male    0.151515   21.113131
Third class  female  0.460784   15.875369
             male    0.150198   12.162695

In [91]:
# Q2 From salaries.csv, build a year-over-year salary report: 
# clean nulls → group by Year + JobTitle 
# → avg pay → unstack Year into columns 
# → add a YoY growth % column for each title. 
# Export to Excel with sheet name "YoY Report".

In [92]:
salary.isna().mean()*100

Id                    0.000000
EmployeeName          0.000000
JobTitle              0.000000
BasePay               0.409676
OvertimePay           0.002691
OtherPay              0.002691
Benefits             24.326961
TotalPay              0.000000
TotalPayBenefits      0.000000
Year                  0.000000
Notes               100.000000
Agency                0.000000
Status              100.000000
dtype: float64

In [93]:
df_q2 = salary[salary['TotalPay']>0].groupby(['Year','JobTitle'])['TotalPay'].mean().reset_index().sort_values(by=['JobTitle','Year'])


In [95]:
df_q2['Prev TotalPay'] = df_q2.groupby(['JobTitle'])['TotalPay'].shift(1)
df_q2['YoY growth pct'] = (df_q2['TotalPay']/df_q2['Prev TotalPay'] - 1)*100
df_q2_unstacked = df_q2.pivot_table(
    index='JobTitle',
    columns='Year',
    values=['TotalPay', 'YoY growth pct']
)
df_q2_unstacked

TotalPay                \
Year                                                    2011          2012   
JobTitle                                                                     
ACCOUNT CLERK                                   44035.664337           NaN   
ACCOUNTANT                                      47429.268000           NaN   
ACCOUNTANT INTERN                               29031.742917           NaN   
ACPO,JuvP, Juv Prob (SFERS)                              NaN           NaN   
ACUPUNCTURIST                                   67594.400000           NaN   
...                                                      ...           ...   
X-RAY LABORATORY AIDE                           52705.880385           NaN   
X-Ray Laboratory Aide                                    NaN  53492.062258   
YOUTH COMMISSION ADVISOR, BOARD OF SUPERVISORS  53632.870000           NaN   
Youth Comm Advisor                                       NaN  57544.730000   
ZOO CURATOR                                     66686.560000           NaN   

                                                                            \
Year                                                    2013          2014   
JobTitle                                                                     
ACCOUNT CLERK                                            NaN           NaN   
ACCOUNTANT                                               NaN           NaN   
ACCOUNTANT INTERN                                        NaN           NaN   
ACPO,JuvP, Juv Prob (SFERS)                              NaN  62290.780000   
ACUPUNCTURIST                                            NaN           NaN   
...                                                      ...           ...   
X-RAY LABORATORY AIDE                                    NaN           NaN   
X-Ray Laboratory Aide                           47992.220588  51211.566857   
YOUTH COMMISSION ADVISOR, BOARD OF SUPERVISORS           NaN           NaN   
Youth Comm Advisor                              35823.295000  36465.910000   
ZOO CURATOR                                              NaN           NaN   

                                               YoY growth pct            
Year                                                     2013      2014  
JobTitle                                                                 
ACCOUNT CLERK                                             NaN       NaN  
ACCOUNTANT                                                NaN       NaN  
ACCOUNTANT INTERN                                         NaN       NaN  
ACPO,JuvP, Juv Prob (SFERS)                               NaN       NaN  
ACUPUNCTURIST                                             NaN       NaN  
...                                                       ...       ...  
X-RAY LABORATORY AIDE                                     NaN       NaN  
X-Ray Laboratory Aide                              -10.281603  6.708059  
YOUTH COMMISSION ADVISOR, BOARD OF SUPERVISORS            NaN       NaN  
Youth Comm Advisor                                 -37.747045  1.793847  
ZOO CURATOR                                               NaN       NaN  

[2155 rows x 6 columns]

In [96]:
df_q2_unstacked.to_excel('YoY_Report.xlsx', sheet_name='YoY Report')

In [ ]:
# Q3 Using the demographics datasets (both combined), join income_group.xlsx 

In [104]:
df_demo1=pd.read_csv(r"C:\Users\user\Desktop\Python1\DataSample\demographic_data.csv")
df_demo2=pd.read_excel(r"C:\Users\user\Desktop\Python1\DataSample\demographic_data_2.xlsx")
df_income=pd.read_excel(r"C:\Users\user\Desktop\Python1\DataSample\income_group.xlsx")
df_demo = pd.concat([df_demo1, df_demo2], ignore_index = True)
df_q3 = pd.merge(df_demo, df_income, on = 'Income Group')
df_q3['Income Group'].astype('category')
df_q3.head()

,Country Name,Country Code,Birth rate,Internet users,Income Group,Income (updated),Income (2020)
0,Aruba,ABW,10.244,78.9,High income,"> 12,695","> 12,535"
1,Afghanistan,AFG,35.253,5.9,Low income,"< 1,046","< 1,035"
2,Angola,AGO,45.985,19.1,Upper middle income,"4,096 -12,695","4,046 -12,535"
3,Albania,ALB,12.877,57.2,Upper middle income,"4,096 -12,695","4,046 -12,535"
4,United Arab Emirates,ARE,11.044,88.0,High income,"> 12,695","> 12,535"


In [105]:
# and produce a summary pivot: rows = Income Group, columns = [avg Birth rate, avg Internet users, country count]. 
# Apply memory optimization (convert Income Group to category dtype).
pd.pivot_table(
    data = df_q3,
    index = 'Income Group',
    aggfunc = {'Birth rate':'mean','Internet users': 'mean','Country Name':'size'}
    )

,Birth rate,Country Name,Internet users
Income Group,,,
High income,12.746765,68,74.559848
Low income,36.462839,31,8.869355
Lower middle income,26.120725,51,23.786331
Upper middle income,18.672469,49,41.397410


In [ ]:
# Q4 From restaurant.csv, write a fully method-chained pipeline that: 
# filters weekends 
# → adds tip_pct 
# → groups by day + smoker 
# → aggregates total revenue and avg tip_pct 
# → sorts by total revenue desc 
# → saves to CSV.

In [106]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [107]:
def save_csv(df, path):
    df.to_csv(path, index=False)
    return df

In [108]:
rest = pd.read_csv(r"C:\Users\user\Desktop\Python1\DataSample\restaurant.csv")
rest.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [121]:
# filters weekends 
# → adds tip_pct 
# → groups by day + smoker 
# → aggregates total revenue and avg tip_pct 
# → sorts by total revenue desc 
# → saves to CSV.
result_q4 = (
    rest[rest['day'].isin(['Sun','Sat'])]
    .assign(
        tip_pct = lambda x: (x['tip']/x['total_bill']*100).round(2)
    )
    .groupby(['day','smoker'])
    .agg(total_revenue = ('total_bill','sum'),avg_tip_pct = ('tip_pct','mean'))
    .reset_index()
    .sort_values(by = 'total_revenue',ascending = False)
    .pipe(save_csv, 'weekend_revenue.csv')
)
result_q4

,day,smoker,total_revenue,avg_tip_pct
2,Sun,No,1168.88,16.011228
1,Sat,Yes,893.62,14.790000
0,Sat,No,884.78,15.804222
3,Sun,Yes,458.28,18.724737
